### Encode and Decode

- While the Unicode standard defines a mapping from characters to code points (integers), it’s impractical to train tokenizers directly on Unicode code points, since the vocabulary would be prohibitively large (around 150K items) and sparse (since many characters are quite rare).
- Instead, we’ll use a Unicode encoding, which converts a Unicode character into a sequence of bytes.

| Bytes | Pattern                                  | Useful bits | Unicode range        |
| ----- | ---------------------------------------- | ----------- | -------------------- |
| 1     | (0xxxxxxx)                               | 7           | (U+0000)–(U+007F)    |
| 2     | (110xxxxx 10xxxxxx)                     | 11          | (U+0080)–(U+07FF)    |
| 3     | (1110xxxx 10xxxxxx 10xxxxxx)           | 16          | (U+0800)–(U+FFFF)    |
| 4     | (11110xxx 10xxxxxx 10xxxxxx 10xxxxxx) | 21          | (U+10000)–(U+10FFFF) |


#### Encode 
To encode a Unicode string into UTF-8, we can use the encode() function in Python.

In [16]:
test_string = "hello! こんにちは!"
utf8_encoded = test_string.encode("utf-8")

In [18]:
# This will print the UTF-8 encoded bytes representation of the string
# it will be printed in hexadecimal format, starting with b' and ending with '. 
# it chars will be represented as \x followed by two hexadecimal digits, which is the standard way to represent bytes in Python.
# For example, the character 'こ' will be represented as \xe3\x81\x93 in UTF-8 encoding, which corresponds to the hexadecimal values of the bytes that make up the UTF-8 encoding of that character.
# but for ASCII characters like 'h', 'e', 'l', 'o', '!' and space, they will be represented as their ASCII values in hexadecimal, which are \x68, \x65, \x6c, \x6f, \x21 and \x20 respectively.
print(utf8_encoded) 
print(type(utf8_encoded))

b'hello! \xe3\x81\x93\xe3\x82\x93\xe3\x81\xab\xe3\x81\xa1\xe3\x81\xaf!'
<class 'bytes'>


In [19]:
print(list(utf8_encoded)) # Get the byte values for the encoded string (integers from 0 to 255).

[104, 101, 108, 108, 111, 33, 32, 227, 129, 147, 227, 130, 147, 227, 129, 171, 227, 129, 161, 227, 129, 175, 33]


In [22]:
utf8_encoded.decode("utf-8") # Decode the UTF-8 bytes back to a string.

'hello! こんにちは!'

In [24]:
from collections import Counter

# -----------------------------
# Step 1: Helpers
# -----------------------------

def to_bytes(text):
    return list(text.encode("utf-8"))

def get_pairs(tokens):
    return Counter(zip(tokens, tokens[1:]))

def merge_pair(tokens, pair_to_merge, new_token):
    new_tokens = []
    i = 0
    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == pair_to_merge:
            new_tokens.append(new_token)
            i += 2
        else:
            new_tokens.append(tokens[i])
            i += 1
    return new_tokens

# -----------------------------
# Step 2: Training (learn merges)
# -----------------------------

text = "hello hello 😊"
tokens = to_bytes(text)

# Base vocab: bytes
vocab = {i: bytes([i]) for i in range(256)}

merges = {}  # new_token -> (a, b)

print("Initial tokens:", tokens)

num_merges = 5

for step in range(num_merges):
    pairs = get_pairs(tokens)
    if not pairs:
        break

    most_common = pairs.most_common(1)[0][0]
    new_token = max(vocab.keys()) + 1  # safe new id

    print(f"\nStep {step+1}")
    print("Most frequent pair:", most_common, "-> new token:", new_token)

    # store merge rule
    merges[new_token] = most_common

    # build vocab entry (IMPORTANT)
    a, b = most_common
    vocab[new_token] = vocab[a] + vocab[b]

    # apply merge
    tokens = merge_pair(tokens, most_common, new_token)

    print("Tokens:", tokens)
    print("New vocab entry:", new_token, "->", vocab[new_token])

print("\nFinal tokens after training:", tokens)

# -----------------------------
# Step 3: Encoding function
# -----------------------------

def encode(text, merges):
    tokens = to_bytes(text)

    while True:
        pairs = get_pairs(tokens)
        if not pairs:
            break

        # pick first applicable merge (same order as training)
        merge_applied = False
        for new_token, pair in merges.items():
            if pair in pairs:
                tokens = merge_pair(tokens, pair, new_token)
                merge_applied = True
                break

        if not merge_applied:
            break

    return tokens

# -----------------------------
# Step 4: Decoding function
# -----------------------------

def decode(tokens, vocab):
    byte_seq = b"".join(vocab[t] for t in tokens)
    return byte_seq.decode("utf-8")

# -----------------------------
# Step 5: Test encode/decode
# -----------------------------

encoded = encode("hello hello 😊", merges)
print("\nEncoded:", encoded)

decoded = decode(encoded, vocab)
print("Decoded:", decoded)

Initial tokens: [104, 101, 108, 108, 111, 32, 104, 101, 108, 108, 111, 32, 240, 159, 152, 138]

Step 1
Most frequent pair: (104, 101) -> new token: 256
Tokens: [256, 108, 108, 111, 32, 256, 108, 108, 111, 32, 240, 159, 152, 138]
New vocab entry: 256 -> b'he'

Step 2
Most frequent pair: (256, 108) -> new token: 257
Tokens: [257, 108, 111, 32, 257, 108, 111, 32, 240, 159, 152, 138]
New vocab entry: 257 -> b'hel'

Step 3
Most frequent pair: (257, 108) -> new token: 258
Tokens: [258, 111, 32, 258, 111, 32, 240, 159, 152, 138]
New vocab entry: 258 -> b'hell'

Step 4
Most frequent pair: (258, 111) -> new token: 259
Tokens: [259, 32, 259, 32, 240, 159, 152, 138]
New vocab entry: 259 -> b'hello'

Step 5
Most frequent pair: (259, 32) -> new token: 260
Tokens: [260, 260, 240, 159, 152, 138]
New vocab entry: 260 -> b'hello '

Final tokens after training: [260, 260, 240, 159, 152, 138]

Encoded: [260, 260, 240, 159, 152, 138]
Decoded: hello hello 😊
